# Notebook Setup

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import scipy.stats as stats

# Portfolio Setup

In [2]:
tickers = ['AAPL', 'TSLA', 'MSFT', 'META', 'AMZN', 'GOOGL', 'NVDA'] #Ticker der MAG7 
start_capital = 100000

In [3]:
data = yf.download(tickers, start="2016-04-01", end="2026-04-01")['Close']

[*********************100%***********************]  7 of 7 completed


In [4]:
#Berechnung der täglichen Renditen (diskret und logarithmisch)
returns_discrete = data.pct_change().dropna()
returns_log = np.log(data / data.shift(1)).dropna()

#Portfolio-Gewichtung
weights = np.array([1/len(tickers)] * len(tickers)) #1/7 Gewichtung für jedes Asset

#Berechnung der Portfolio-Renditen
portfolio_returns_discrete = returns_discrete.dot(weights)
portfolio_returns_log = returns_log.dot(weights)

Berechnung der täglichen relativen Renditen macht es möglich, dass Daten einer Normalverteilung angenähert werden können. Werden benötigt,um Erwartungswert und Volatilität zu brechnen

In [5]:
portfolio_returns_discrete.tail()

Date
2026-03-25    0.007628
2026-03-26   -0.031970
2026-03-27   -0.027635
2026-03-30   -0.001346
2026-03-31    0.045284
dtype: float64

# Brechnung 1-Tages-Risikomodelle 

Value at Risk und expected Shortfall für Gaussian, lognorm und historische Modelle

Formel Gaussian Value at Risk: VaR = -(Erwartungswert(portfolio)+Z-Wert(alpha)*Volatilität)*Kapital 

In [6]:
# 1. Historisches Risiko (VaR & ES)
def calculate_historical_risk_1d(returns, capital, var_level):
    alpha = var_level
    clean_returns = returns.dropna()
    
    # VaR in % berechnen
    var_pct = np.percentile(clean_returns, alpha * 100)
    
    # ES berechnen (nutzt den exakten var_pct von oben als Filter)
    tail_returns = clean_returns[clean_returns <= var_pct]
    es_pct = np.mean(tail_returns)
    
    # Gibt beide Werte gleichzeitig zurück
    return capital * abs(var_pct), capital * abs(es_pct) #eigentlich negativer Wert, annahme, dass Funktion von VaR und ES bekannt ist


# 2. Gaußsches Risiko (VaR & ES)
def calculate_gaussian_risk_1d(returns, capital, var_level):
    alpha = var_level
    clean_returns = returns.dropna()
    
    mu = np.mean(clean_returns) #Erwartungswert der Renditen
    sigma = np.std(clean_returns) #Standardabweichung der Renditen
    z_score = stats.norm.ppf(alpha) #Z-Wert 
    
    # VaR in % berechnen
    var_pct = mu + (z_score * sigma)
    
    # ES berechnen
    es_pct = mu - sigma * (stats.norm.pdf(z_score) / alpha)
    
    return capital * abs(var_pct), capital * abs(es_pct)


# 3. Lognormales Risiko (VaR & ES)
def calculate_lognormal_risk_1d(log_returns, capital, var_level):
    alpha = var_level
    clean_returns = log_returns.dropna()
    
    mu_log = np.mean(clean_returns)
    sigma_log = np.std(clean_returns)
    z_score = stats.norm.ppf(alpha)
    
    # VaR in % berechnen
    log_worst_case = mu_log + (z_score * sigma_log)
    var_pct = np.exp(log_worst_case) - 1 #Umwandlung mit exp, da log
    
    # ES berechnen
    expected_value_lognorm = np.exp(mu_log + (sigma_log**2) / 2) #Erwartungswert der Lognormalverteilung
    tail_factor = stats.norm.cdf(z_score - sigma_log) / alpha #Berechnet den Flächenanteil des Expected Shortfalls im logarithmischen "Tail" (Extrembereich)
    es_pct = (expected_value_lognorm * tail_factor) - 1
    
    return capital * abs(var_pct), capital * abs(es_pct)

In [7]:
# var_level (alpha) festlegen
alpha = 0.05 

hist_var, hist_es = calculate_historical_risk_1d(portfolio_returns_discrete, start_capital, alpha)
gauss_var, gauss_es = calculate_gaussian_risk_1d(portfolio_returns_discrete, start_capital, alpha)
lognorm_var, lognorm_es = calculate_lognormal_risk_1d(portfolio_returns_log, start_capital, alpha)

# Ausgabe
print(f"Historisch: VaR = {hist_var:,.2f} $, ES = {hist_es:,.2f} $")
print(f"Gaußsch:    VaR = {gauss_var:,.2f} $, ES = {gauss_es:,.2f} $")
print(f"Lognormal:  VaR = {lognorm_var:,.2f} $, ES = {lognorm_es:,.2f} $")

Historisch: VaR = 3,000.60 $, ES = 4,250.44 $
Gaußsch:    VaR = 2,853.81 $, ES = 3,613.13 $
Lognormal:  VaR = 2,846.29 $, ES = 3,579.73 $


# Skalierung auf Zeiträume: 1 Jahr, 5 Jahre und 10 Jahre 

In [8]:
def calculate_historical_risk(log_returns, capital, var_level, days):
    if days == 1:
        rolling_discrete = np.exp(log_returns.dropna()) - 1 #Kein Window, da nur 1 Tag betrachtet wird, auch keine Log-Returns, da diskrete Renditen für 1 Tag ausreichend
    else:
        rolling_log = log_returns.rolling(window=days).sum().dropna() #Rolling Window über die Log-Returns, da für mehrere Tage die Log-Returns additiv sind (Renditen werden multipliziert, Logarithmus macht sie additiv)
        
        if len(rolling_log) < 100: #Sicherheitscheck, um zu verhindern, dass zu wenige Datenpunkte für die Berechnung genutzt werden
            return np.nan, np.nan
            
        rolling_discrete = np.exp(rolling_log) - 1 #Umwandlung der rollenden Log-Returns in diskrete Renditen

    var_pct = np.percentile(rolling_discrete, var_level * 100)  #VaR in % berechnen mit 5. Perzentil der rollenden diskreten Renditen
    tail_returns = rolling_discrete[rolling_discrete <= var_pct] #Filtert die Renditen, die im "Tail" liegen (also kleiner oder gleich dem berechneten VaR)
    es_pct = np.mean(tail_returns) #ES berechnen als Durchschnitt der Renditen im Tail (also der Renditen, die den VaR überschreiten)

    return capital * var_pct, capital * es_pct #Berechnung der abosluten VaR und ES

In [9]:
def calculate_gaussian_risk(returns, capital, var_level, days):
    clean_returns = returns.dropna()
    
    mu_1d = np.mean(clean_returns) #Erwartungswert der Renditen Tagesbasis
    sigma_1d = np.std(clean_returns) #Volatilität der Renditen Tagesbasis
    
    mu_nd = mu_1d * days #Erwarungswert für n Tage (Renditen sind addititv)
    sigma_nd = sigma_1d * np.sqrt(days) #Volatilität für n Tage mal Wurzerl aus T (hier Anzahl der Tage)
    
    z_score = stats.norm.ppf(var_level) #Z-Wert für Normalverteilung
    
    var_pct = mu_nd + (z_score * sigma_nd) 
    es_pct = mu_nd - sigma_nd * (stats.norm.pdf(z_score) / var_level)
    
    return capital * var_pct, capital * es_pct

In [10]:
def calculate_lognormal_risk(log_returns, capital, var_level, days):
    clean_returns = log_returns.dropna()
    
    mu_log_1d = np.mean(clean_returns)
    sigma_log_1d = np.std(clean_returns)
    
    mu_log_nd = mu_log_1d * days
    sigma_log_nd = sigma_log_1d * np.sqrt(days)
    
    z_score = stats.norm.ppf(var_level)
    
    log_worst_case = mu_log_nd + (z_score * sigma_log_nd)
    var_pct = np.exp(log_worst_case) - 1 #Umwandlung mit exp, da log
    
    expected_value_lognorm = np.exp(mu_log_nd + (sigma_log_nd**2) / 2) #Erwartungswert der Lognormalverteilung
    tail_factor = stats.norm.cdf(z_score - sigma_log_nd) / var_level
    es_pct = (expected_value_lognorm * tail_factor) - 1
    
    return capital * var_pct, capital * es_pct

In [11]:
# Parameter definieren
alpha_level = 0.05

# Zeiträume in Handelstagen (1 Jahr, 5 Jahre, 10 Jahre)
horizons = {
    "1 Jahr": 252,
    "5 Jahre": 1260,
    "10 Jahre": 2520
}

print(f"{' Zeitraum ':=^75}")

for label, days in horizons.items():
    print(f"\n--- HORIZONT: {label} ({days} Handelstage) ---")
    
    # Berechnungen (Aufruf der neuen, universellen Funktionen)
    h_var, h_es = calculate_historical_risk(portfolio_returns_log, start_capital, alpha_level, days)
    g_var, g_es = calculate_gaussian_risk(portfolio_returns_discrete, start_capital, alpha_level, days)
    l_var, l_es = calculate_lognormal_risk(portfolio_returns_log, start_capital, alpha_level, days)
    
    # Ausgabe formatieren
    print(f"{'Modell':<15} | {'Value at Risk':>15} | {'Expected Shortfall':>20}")
    print("-" * 56)
    
    # Für das historische Modell müssen wir abfangen, wenn es zu wenig Daten gibt (wie bei 10 Jahren)
    if np.isnan(h_var):
        print(f"{'Historisch':<15} | {'Zu wenig Daten':>15} | {'Zu wenig Daten':>20}")
    else:
        print(f"{'Historisch':<15} | {h_var:>13,.2f} $ | {h_es:>18,.2f} $")
        
    print(f"{'Gaußsch':<15} | {g_var:>13,.2f} $ | {g_es:>18,.2f} $")
    print(f"{'Lognormal':<15} | {l_var:>13,.2f} $ | {l_es:>18,.2f} $")

================================ Zeitraum =================================

--- HORIZONT: 1 Jahr (252 Handelstage) ---
Modell          |   Value at Risk |   Expected Shortfall
--------------------------------------------------------
Historisch      |    -22,906.45 $ |         -35,858.78 $
Gaußsch         |    -13,385.19 $ |         -25,439.06 $
Lognormal       |    -19,112.50 $ |         -27,913.16 $

--- HORIZONT: 5 Jahre (1260 Handelstage) ---
Modell          |   Value at Risk |   Expected Shortfall
--------------------------------------------------------
Historisch      |    149,372.02 $ |         134,584.62 $
Gaußsch         |     64,218.92 $ |          37,265.64 $
Lognormal       |     28,681.15 $ |             814.40 $

--- HORIZONT: 10 Jahre (2520 Handelstage) ---
Modell          |   Value at Risk |   Expected Shortfall
--------------------------------------------------------
Historisch      |  Zu wenig Daten |       Zu wenig Daten
Gaußsch         |    190,588.84 $ |         15

In [12]:
benchmark_world_data = yf.download("VT", start="2016-04-01", end="2026-04-01")['Close'] #VT (Vanguard Total World Stock ETF) 
benchmark_risk_free_data = yf.download("^TNX", start="2016-04-01", end="2026-04-01")['Close'] #^TNX (10-Year Treasury Note Yield)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
